In [1]:
# !pip install torch torchvision torchaudio transformers accelerate bitsandbytes --quiet
# !pip install -U bitsandbytes

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
import os
import random
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve)

import warnings
warnings.filterwarnings("ignore")

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [3]:
drive.mount('/content/drive')

drive_folder = "/content/drive/MyDrive/Aireen Y4S1/WIE3007 DATA MINING AND WAREHOUSING/Group Assignment 15%"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Setup huggingface token
from google.colab import userdata
HUGGINGFACE_TOKEN = userdata.get("HF_TOKEN")

from huggingface_hub import login
login(token=HUGGINGFACE_TOKEN)

In [5]:
preprocessed_df = pd.read_csv(os.path.join(drive_folder, "preprocessed_data.csv"))

# Feature Engineering

In [6]:
preprocessed_df.drop(columns=['Sentiment_Score'], inplace=True)
preprocessed_df.drop(columns=['LoanPurposeDescription'], inplace=True)

In [7]:
preprocessed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 47 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   CustomerID                           1000 non-null   int64  
 1   Age                                  1000 non-null   int64  
 2   Education                            1000 non-null   float64
 3   EmploymentLengthYears                1000 non-null   int64  
 4   MonthlyIncome                        1000 non-null   float64
 5   Dependents                           1000 non-null   int64  
 6   CreditScore                          1000 non-null   int64  
 7   ExistingLoans                        1000 non-null   int64  
 8   MonthlyDebt                          1000 non-null   float64
 9   YearsWithBank                        1000 non-null   int64  
 10  HasSavingsAccount                    1000 non-null   int64  
 11  HasCheckingAccount             

In [8]:
X = preprocessed_df.drop("LoanDefault", axis=1)
y = preprocessed_df["LoanDefault"]

numerical_cols = ["Age", "EmploymentLengthYears", "MonthlyIncome", "Dependents", "CreditScore", "ExistingLoans", "MonthlyDebt", "YearsWithBank", "LoanAmount", "LoanTermMonths", "InterestRate" ]

In [9]:
X_normalized = X.copy()

standard_scaler = StandardScaler()

X_normalized[numerical_cols] = standard_scaler.fit_transform(X[numerical_cols])

print("Normalization using StandardScaler")
print(f"Shape: {X_normalized.shape}")

# See before and after
print("\nBefore Normalization:")
print(X[numerical_cols].head(10))
print("\nAfter Normalization:")
print(X_normalized[numerical_cols].head(10))

Normalization using StandardScaler
Shape: (1000, 46)

Before Normalization:
   Age  EmploymentLengthYears  MonthlyIncome  Dependents  CreditScore  \
0   59                     11    9067.159786           2          770   
1   51                     24    5068.058742           1          698   
2   24                      0    2557.541824           0          480   
3   25                      0    7535.097299           0          774   
4   25                      5    4798.465251           2          590   
5   18                      0    2715.312586           0          576   
6   26                      2    8468.695614           2          648   
7   23                      0    2079.184097           3          581   
8   46                      8    9386.927084           0          688   
9   39                     13    9241.197498           2          655   

   ExistingLoans  MonthlyDebt  YearsWithBank    LoanAmount  LoanTermMonths  \
0              3  1583.065500             

# Model Development

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42, stratify=y)

## Overfitting Model

In [11]:
# Train overfitting model
knn_overfit = KNeighborsClassifier(n_neighbors=1)
knn_overfit.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=1)

In [12]:
# Evaluate overfitting model
y_train_pred_overfit = knn_overfit.predict(X_train)
y_test_pred_overfit = knn_overfit.predict(X_test)

train_accuracy_overfit = accuracy_score(y_train, y_train_pred_overfit)
test_accuracy_overfit = accuracy_score(y_test, y_test_pred_overfit)

print(f"OVERFITTING MODEL:")
print(f"Train Accuracy: {train_accuracy_overfit:.4f}")
print(f"Test Accuracy: {test_accuracy_overfit:.4f}")
print(f"Overfitting Gap: {(train_accuracy_overfit - test_accuracy_overfit):.4f}")

print("Classification Report:")
print(classification_report(y_test, y_test_pred_overfit))

OVERFITTING MODEL:
Train Accuracy: 1.0000
Test Accuracy: 0.6200
Overfitting Gap: 0.3800
Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.54      0.59       100
           1       0.60      0.70      0.65       100

    accuracy                           0.62       200
   macro avg       0.62      0.62      0.62       200
weighted avg       0.62      0.62      0.62       200



## Optimal Model

In [13]:
# Hyperparameter tuning

In [14]:
# Train optimal model

# Model Evaluation and Interpretation